In [2]:
# 6-21-2026

In [4]:
import pandas as pd
import joblib
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from scipy.stats import spearmanr


In [81]:
# adjust hyperparameters by experimenting with a single domain. same hyperparams will be used for each domain (identical model)

In [92]:
domain_id = "20"

# load raw train and test splits for this domain, it's around average for domain dataset size
X_train = pd.read_csv(f"train_X/domain_{domain_id}.csv")
y_train = pd.read_csv(f"train_y/domain_{domain_id}.csv")["log_ba"]
X_test = pd.read_csv(f"test_X/domain_{domain_id}.csv")
y_test = pd.read_csv(f"test_y/domain_{domain_id}.csv")["log_ba"]

In [93]:
scaler = joblib.load(f"scalers/domain_{domain_id}.joblib")

In [94]:
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [95]:
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_scaled, y_train, test_size=0.2, random_state=5
) # make eval set to find strong hyperparams

In [106]:
model = XGBRegressor(
    n_estimators=1000,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=1.0,
    reg_lambda=2.0,
    random_state=5,
    n_jobs=-1,
    eval_metric="rmse",
    early_stopping_rounds=30
)

In [107]:
model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)

,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.8
,device,None
,early_stopping_rounds,30
,enable_categorical,False
,eval_metric,'rmse'


In [108]:
print(f"best iteration: {model.best_iteration}")

best iteration: 999


In [109]:
train_pred = model.predict(X_train_scaled)
test_pred = model.predict(X_test_scaled)

train_mse = mean_squared_error(y_train, train_pred)
test_mse = mean_squared_error(y_test, test_pred)
train_r2 = r2_score(y_train, train_pred)
test_r2 = r2_score(y_test, test_pred)

In [110]:
print(f"train mse: {train_mse:.4f}, test mse: {test_mse:.4f}")
print(f"train r2: {train_r2:.4f}, test r2: {test_r2:.4f}")

train mse: 0.7284, test mse: 0.7835
train r2: 0.1982, test r2: 0.1437


In [111]:
from scipy.stats import spearmanr

# spearman correlation, robust to outliers and scale, tests if relative ordering is captured
train_spearman, _ = spearmanr(y_train, train_pred)
test_spearman, _ = spearmanr(y_test, test_pred)

print(f"train spearman: {train_spearman:.4f}, test spearman: {test_spearman:.4f}")

train spearman: 0.4792, test spearman: 0.4285


In [66]:
# time to make T for xgb. using hyperparams above. T(i,j) includes BOTH r2 and spearman, will choose best one later

In [5]:
XGB_PARAMS = {
    "n_estimators": 1000,
    "max_depth": 4,
    "learning_rate": 0.05,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "reg_alpha": 1.0,
    "reg_lambda": 2.0,
    "random_state": 10,
    "n_jobs": -1,
    "early_stopping_rounds": 30,
    "eval_metric": "rmse"
}

In [6]:
import os
import glob
import pandas as pd
import joblib
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from scipy.stats import spearmanr

In [7]:
domain_files = glob.glob("train_X/domain_*.csv")
domain_ids = sorted(
    int(os.path.basename(f).replace("domain_", "").replace(".csv", ""))
    for f in domain_files
)
print(f"found {len(domain_ids)} domains")

found 34 domains


In [8]:
models = {}
scalers = {}

for domain_id in domain_ids:
    X_train = pd.read_csv(f"train_X/domain_{domain_id}.csv")
    y_train = pd.read_csv(f"train_y/domain_{domain_id}.csv")["log_ba"]

    scaler = joblib.load(f"scalers/domain_{domain_id}.joblib")
    X_train_scaled = scaler.transform(X_train)

    # make  a validation split out of train only, for early stopping
    X_tr, X_val, y_tr, y_val = train_test_split(
        X_train_scaled, y_train, test_size=0.2, random_state=5
    )

    model = XGBRegressor(**XGB_PARAMS)
    model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)

    models[domain_id] = model
    scalers[domain_id] = scaler

    print(f"domain {domain_id} done, best iteration: {model.best_iteration}")
# takes ~3min

domain 0 done, best iteration: 999
domain 1 done, best iteration: 234
domain 2 done, best iteration: 982
domain 4 done, best iteration: 629
domain 5 done, best iteration: 995
domain 6 done, best iteration: 135
domain 7 done, best iteration: 51
domain 8 done, best iteration: 268
domain 11 done, best iteration: 999
domain 12 done, best iteration: 652
domain 13 done, best iteration: 850
domain 16 done, best iteration: 510
domain 18 done, best iteration: 28
domain 19 done, best iteration: 272
domain 20 done, best iteration: 999
domain 21 done, best iteration: 788
domain 22 done, best iteration: 999
domain 23 done, best iteration: 506
domain 25 done, best iteration: 668
domain 26 done, best iteration: 280
domain 27 done, best iteration: 442
domain 28 done, best iteration: 295
domain 29 done, best iteration: 355
domain 30 done, best iteration: 101
domain 32 done, best iteration: 162
domain 33 done, best iteration: 167
domain 36 done, best iteration: 71
domain 37 done, best iteration: 999
dom

In [9]:
# preload all test sets once, dont rereading csvs 34 times per domain
test_X_raw = {}
test_y = {}

for domain_id in domain_ids:
    test_X_raw[domain_id] = pd.read_csv(f"test_X/domain_{domain_id}.csv")
    test_y[domain_id] = pd.read_csv(f"test_y/domain_{domain_id}.csv")["log_ba"]

In [10]:
# T(i, j): i is source domain (model trained on i), j is target domain (evaluated on j)
T_r2 = pd.DataFrame(index=domain_ids, columns=domain_ids, dtype=float)
T_spearman = pd.DataFrame(index=domain_ids, columns=domain_ids, dtype=float)

In [11]:
for i in domain_ids:
    model_i = models[i]
    scaler_i = scalers[i]

    for j in domain_ids:
        # apply source domain's scaler to target domain's raw test X, not target's own scaler
        X_test_scaled = scaler_i.transform(test_X_raw[j])
        y_true = test_y[j]

        preds = model_i.predict(X_test_scaled)

        T_r2.loc[i, j] = r2_score(y_true, preds)
        T_spearman.loc[i, j], _ = spearmanr(y_true, preds)

    print(f"finished evaluating source domain {i} against all targets")

finished evaluating source domain 0 against all targets
finished evaluating source domain 1 against all targets
finished evaluating source domain 2 against all targets
finished evaluating source domain 4 against all targets
finished evaluating source domain 5 against all targets
finished evaluating source domain 6 against all targets
finished evaluating source domain 7 against all targets
finished evaluating source domain 8 against all targets
finished evaluating source domain 11 against all targets
finished evaluating source domain 12 against all targets
finished evaluating source domain 13 against all targets
finished evaluating source domain 16 against all targets
finished evaluating source domain 18 against all targets
finished evaluating source domain 19 against all targets
finished evaluating source domain 20 against all targets
finished evaluating source domain 21 against all targets
finished evaluating source domain 22 against all targets
finished evaluating source domain 23 ag

In [12]:
T_r2

,0,1,2,4,5,6,7,8,11,12,...,32,33,36,37,38,39,45,46,47,49
0,0.082404,0.036398,-0.189292,-0.058032,0.009610,-0.018824,-0.025163,-0.271086,-0.221360,-0.053864,...,-0.018590,-0.003790,-0.003967,-0.124740,-0.017629,-0.199646,-0.166572,-0.023717,-0.137179,-0.024580
1,-0.039496,0.066218,-2.009305,-0.117947,0.012791,-0.261016,-0.105569,-0.201914,-0.377985,-0.102628,...,-0.079695,-0.252915,-0.161178,-0.222769,-0.085094,-0.231542,-0.390769,-0.998980,-0.251673,-0.110201
2,-0.107104,-0.165887,0.075630,-0.158890,-0.149034,-0.141825,-0.129815,-0.097988,-0.050250,-0.106770,...,-0.029591,-0.111490,-0.242448,-0.144484,-0.183739,-0.218916,-0.111903,-0.294891,-0.091894,-0.169791
4,-0.009746,-0.010566,-0.296821,0.083725,-0.012926,0.011441,-0.106578,-0.287240,-0.323501,-0.078233,...,-0.010084,-0.002242,-0.047218,-0.324396,-0.026150,-0.056901,-0.275495,0.070813,-0.031414,-0.058628
5,-0.337814,-0.190564,-2.066483,-0.169526,0.125379,-0.198547,-0.130695,-0.090194,-0.483618,-0.087110,...,-0.589283,-1.257822,-0.353142,-0.215237,-0.205053,-0.467720,-0.431205,-0.163518,-0.271608,-0.255946
6,-0.092642,-0.136124,-0.334040,0.000585,-0.136664,0.059770,-0.129389,-0.261298,-0.542087,-0.365457,...,-0.093770,-0.042144,-0.043202,-0.414096,-0.108602,-0.196543,-0.703503,0.074989,-0.125554,-0.030974
7,-0.018564,-0.019254,-0.162730,-0.002792,-0.025592,0.002489,0.028897,-0.036043,-0.119735,-0.047231,...,-0.005339,0.024031,0.015080,-0.128155,0.003578,-0.011232,-0.133628,0.016707,-0.061345,-0.004572
8,-0.292899,-0.141115,-1.205311,-0.386683,-0.202782,-0.253291,-0.204251,0.092098,-1.405107,-0.325272,...,-0.230194,-0.415000,-0.376312,-0.869897,-0.419306,-0.668299,-1.102400,-0.597216,-0.418444,-0.184690
11,-0.026846,-0.086539,-0.210739,-0.143707,0.004761,-0.112766,-0.044205,-0.113807,0.149890,-0.162046,...,0.009453,-0.091958,-0.094828,0.076045,-0.077498,-0.396077,-0.096500,-0.119455,-0.242308,-0.093096
12,-0.181581,-0.043187,-1.374847,-0.223946,-0.085980,-0.113077,-0.194358,-0.724191,-0.506579,0.145272,...,-0.433989,-0.505592,-0.178216,-0.292543,-0.306948,-0.199011,-0.830016,-0.279501,-0.165674,-0.241674


In [13]:
T_spearman

,0,1,2,4,5,6,7,8,11,12,...,32,33,36,37,38,39,45,46,47,49
0,0.340909,0.267183,0.058919,0.061634,0.195116,0.077565,0.144459,0.077099,0.182382,0.065096,...,0.182985,0.199124,0.119656,0.286961,0.081828,-0.070119,0.101100,0.109456,0.055133,0.029534
1,0.144479,0.356439,-0.004450,0.019226,0.188342,0.001954,0.133262,0.059977,0.138771,0.052687,...,0.202479,0.049018,0.048965,0.217383,0.078370,0.024229,0.083000,0.049858,0.039416,-0.015148
2,0.178284,0.180319,0.299617,0.023789,0.163983,0.021422,0.074409,0.143664,0.155447,0.049786,...,0.162703,0.212451,0.095598,0.210661,0.007015,0.077337,0.086238,-0.023865,0.087231,-0.038248
4,0.214375,0.247242,0.046393,0.332562,0.170113,0.186322,0.065848,0.182742,0.095542,0.034397,...,0.159622,0.194794,0.140972,0.263918,0.164294,0.036173,0.060893,0.310646,0.080613,0.048915
5,0.058294,0.090082,0.010361,0.019192,0.406833,0.081617,0.111621,0.179313,0.158728,0.080677,...,0.111890,0.151063,0.038804,0.220161,0.085753,0.026931,0.090576,0.143931,-0.001661,0.035539
6,0.142245,0.217627,0.003595,0.208823,0.126453,0.261848,0.056133,0.174523,0.058929,-0.005014,...,0.075688,0.156501,0.035556,0.241400,0.084012,-0.031058,0.093640,0.295058,0.112769,0.040151
7,0.133016,0.045072,0.084613,0.062888,0.092101,0.082992,0.244994,-0.022968,0.206551,0.007743,...,0.152022,0.219326,0.138865,0.198044,0.207786,-0.022669,0.081149,0.112908,-0.028804,0.059214
8,0.073564,0.158986,-0.043273,-0.012496,0.000387,0.050860,0.061105,0.374431,-0.119477,0.020646,...,0.032126,0.129240,-0.006809,-0.108581,-0.102111,0.053754,-0.027862,0.173907,0.035938,0.008973
11,0.206437,0.160151,0.046533,0.027241,0.216630,0.026120,0.102000,0.116002,0.451521,0.018463,...,0.223940,0.185784,0.083830,0.367883,0.179426,-0.028463,0.134259,0.064131,-0.058619,0.029846
12,0.090474,0.156002,0.000954,0.060133,0.104244,0.069411,0.006661,-0.031128,0.158737,0.431843,...,0.093991,-0.142449,0.023021,0.227597,0.009939,0.064134,0.087512,-0.006709,0.063450,0.015122


In [14]:
# check if T(i,j) variance is driven by domain size rather than real transfer differences
domain_sizes = {d: len(pd.read_csv(f"train_X/domain_{d}.csv")) for d in domain_ids}

row_variance = T_spearman.var(axis=1)  # variance of each source domain's row across all targets
sizes = pd.Series(domain_sizes)

print(pd.concat([sizes.rename("train_size"), row_variance.rename("spearman_row_var")], axis=1).sort_values("train_size"))

    train_size  spearman_row_var
30        1825          0.004740
7         3882          0.005747
33        4878          0.005922
39        5817          0.005904
18        6910          0.007236
19        8102          0.005430
36        8936          0.004752
32        9909          0.007090
6        12588          0.007314
38       13423          0.004301
26       13601          0.005805
46       15258          0.010379
49       15598          0.005302
47       21231          0.009267
29       23199          0.005830
16       23340          0.006708
8        26427          0.008333
28       28653          0.004213
1        32690          0.007043
12       34539          0.009153
23       39176          0.007849
27       46920          0.007539
25       68170          0.010503
21       76967          0.005413
13       84219          0.007929
2        91063          0.005682
22      103595          0.007143
5       128217          0.007571
4       140541          0.006196
20      15

In [15]:
# does variance across SOURCE models for a given TARGET correlate with the target's test set size
target_sizes = {d: len(pd.read_csv(f"test_X/domain_{d}.csv")) for d in domain_ids}

col_variance = T_spearman.var(axis=0)  # variance down each target column, across all 34 source models
sizes_col = pd.Series(target_sizes)

print(pd.concat([sizes_col.rename("test_size"), col_variance.rename("spearman_col_var")], axis=1).sort_values("test_size"))

    test_size  spearman_col_var
30        609          0.004138
7        1294          0.004540
33       1626          0.010909
39       1940          0.005445
18       2304          0.008815
19       2701          0.004557
36       2979          0.003320
32       3303          0.007526
6        4197          0.004808
38       4475          0.009347
26       4534          0.005093
46       5086          0.010011
49       5200          0.003306
47       7078          0.007247
29       7733          0.004893
16       7781          0.008304
8        8809          0.007209
28       9551          0.003098
1       10897          0.006341
12      11513          0.006465
23      13059          0.005136
27      15641          0.009519
25      22724          0.019800
21      25656          0.004980
13      28074          0.008487
2       30355          0.005606
22      34532          0.005559
5       42740          0.006005
4       46847          0.006664
20      51022          0.007976
37      

In [16]:
print("row variance vs train size:", row_variance.corr(sizes))
print("col variance vs test size:", T_spearman.var(axis=0).corr(pd.Series(target_sizes)))

row variance vs train size: 0.5669910698753842
col variance vs test size: 0.2847836950334675


In [17]:
# spearman seems to be the stronger evaluator of domain performance, so will be going with that

In [18]:
T_spearman.to_csv("transfer_matrix_spearman_xgb_10.csv")